# **HACKATHON DE BRIAN SEPULVEDA ROMERO**



El problema a resolver en este Hackathon es sobre el reglamento escolar, el cual este asistente inteligente consulte una base de conocimiento de dicho reglamento antes de responder, reutilizando el reglamento que habia hecho en la practica anteriormente, pero ampliandolo y combinando las técnicas aprendidas en las anteriores MasterClasses las cuales fueron: Llama, prompt engineering y RAG, fine-tuning con LoRA y despliegue en un endpoint.

## **CONFIGURACIÓN DEL ENTORNO**

### **ACTIVAR LA GPU**


In [43]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### **INSTALAR OLLAMA**


In [44]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


### **INICIAR EL SERVIDOR DE OLLAMA**

In [45]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess
import time
import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):
    try:
        requests.get("http://localhost:11434")
        print("Servidor de Ollama listo.")
        break
    except requests.exceptions.ConnectionError:
        time.sleep(1)
else:
    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.14


### **DESCARGAR EL MODELO**

In [46]:
# Descargar el modelo Llama (la primera vez tarda unos minutos)
MODELO = "llama3.1:8b"

!ollama pull {MODELO}
!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


### **INSTALAR LIBRERÍAS**

In [47]:
# Instalar librerías que se necesitan para el Hackathon
!pip install ollama sentence-transformers transformers peft accelerate trl datasets gradio -q

print("Librerías instaladas.")

Librerías instaladas.


### **CLIENTE DE PYTHON**

In [48]:
# Conectar el cliente de Ollama al servidor local
import ollama

client = ollama.Client(host="http://localhost:11434")
print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **BASE DE CONOCIMIENTO (RAG)**
RAG busca primero los fragmentos más relevantes del reglamento (con embeddings y similitud coseno) y después se los entrega al modelo junto con la pregunta, se reutilizaran los fragmentos de la practica Prompt Engineering y Sistemas RAG, aumentandole mas frgamentos y generando los embeddings.

In [49]:
# Definir la base de conocimiento y generar sus embeddings
from sentence_transformers import SentenceTransformer
import numpy as np

documentos = [
    "Los estudiantes deben portar su credencial escolar dentro de las instalaciones "
    "y presentarla cuando sea solicitada por el personal autorizado.",

    "Está prohibido introducir bebidas alcohólicas, sustancias ilegales o cualquier "
    "objeto que represente un riesgo para la comunidad estudiantil.",

    "Los estudiantes deben respetar los horarios de entrada y salida establecidos "
    "por la institución. Los retardos deberán justificarse ante las autoridades "
    "escolares.",

    "Los estudiantes deben mantener un trato respetuoso con compañeros, docentes y "
    "personal administrativo; las faltas de respeto serán sancionadas.",

    "El uso de teléfonos celulares durante clase solo se permite con autorización "
    "del docente.",

    "Se requiere un mínimo de 80% de asistencia para tener derecho a examen ordinario.",

    "La calificación aprobatoria es de 6 en adelante, en una escala de 0 a 10.",

    "Los daños al mobiliario o equipo de la institución deberán ser reparados o "
    "pagados por el responsable.",

    "Para justificar una inasistencia se debe presentar un comprobante en un plazo "
    "de 3 días hábiles.",

    "Está prohibido copiar en los exámenes; la primera falta anula el examen.",
]

modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
embeddings_documentos = modelo_embeddings.encode(documentos)

print("Fragmentos:", len(documentos))
print("Embeddings generados:", embeddings_documentos.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fragmentos: 10
Embeddings generados: (10, 384)


### **BUSCAR LOS FRAGMENTOS MÁS RELEVANTES**

En lugar de devolver solo el fragmento más parecido, devolvemos los dos mas parecidos. Así una pregunta que toca dos reglas no pierde información.

In [50]:
# Definir la función que recupera los k fragmentos más relevantes
from sklearn.metrics.pairwise import cosine_similarity

K = 2

def buscar_fragmentos(pregunta, k=K):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = cosine_similarity(embeddings_documentos, embedding_pregunta).flatten()
    indices = similitudes.argsort()[::-1][:k]
    return [documentos[i] for i in indices]

# Prueba rápida
pregunta = "¿Qué pasa si llego tarde a la escuela?"
for f in buscar_fragmentos(pregunta):
    print("-", f)

- Los estudiantes deben respetar los horarios de entrada y salida establecidos por la institución. Los retardos deberán justificarse ante las autoridades escolares.
- Para justificar una inasistencia se debe presentar un comprobante en un plazo de 3 días hábiles.


## **PROMPTING: ASISTENTE RAG CON LLAMA 8B**

Estrategia de prompting: **instrucciones restrictivas** (responder solo con el reglamento), **respuesta de rechazo fija** cuando la información no está y **few-shot** con dos ejemplos del formato esperado.

In [51]:
# Definir el asistente RAG con prompt restrictivo y few-shot
MENSAJE_NO_SABE = "No encontré esa información en el reglamento."

def responder_rag(pregunta, k=K):
    contexto = "\n".join(f"- {f}" for f in buscar_fragmentos(pregunta, k))
    prompt = f"""Eres un asistente del reglamento escolar. Responde usando SOLO el reglamento de abajo.
Si el reglamento no cubre la pregunta, responde exactamente: "{MENSAJE_NO_SABE}"

Ejemplo 1
Pregunta: ¿Puedo traer una bebida alcohólica?
Respuesta: No, está prohibido introducir bebidas alcohólicas a la institución.

Ejemplo 2
Pregunta: ¿Cuánto cuesta la cafetería?
Respuesta: {MENSAJE_NO_SABE}

Reglamento:
{contexto}

Pregunta: {pregunta}
Respuesta breve (máximo 2 oraciones)."""
    r = client.chat(model=MODELO, messages=[{"role": "user", "content": prompt}])
    return r.message.content.strip()

print(responder_rag("¿Qué pasa si llego tarde a la escuela?"))
print(responder_rag("¿Hay clases de natación?"))

Los estudiantes deben respetar los horarios de entrada y salida establecidos por la institución. Los retardos deberán justificarse ante las autoridades escolares.
No encontré esa información en el reglamento.


## **EVALUACIÓN: CONJUNTO DE PREGUNTAS**

In [53]:
# Definir el conjunto de evaluación y la función evaluar
preguntas_eval = [
    {"p": "¿Qué pasa si llego tarde a la escuela?",              "clave": ["justific"]},
    {"p": "¿Puedo meter sustancias ilegales al plantel?",        "clave": ["prohib"]},
    {"p": "¿Es obligatorio llevar la credencial?",               "clave": ["credencial", "portar"]},
    {"p": "¿Qué porcentaje de asistencia me piden?",             "clave": ["80"]},
    {"p": "¿Cuál es la calificación mínima para aprobar?",       "clave": ["6"]},
    {"p": "¿Puedo contestar mensajes en clase con mi celular?",  "clave": ["autoriz"]},
    {"p": "Si dañé una butaca, ¿quién la paga?",                 "clave": ["repar", "pag"]},
    {"p": "¿En cuántos días entrego un justificante médico?",    "clave": ["3"]},
    {"p": "¿Qué sanción hay por copiar en un examen?",           "clave": ["anula"]},
    {"p": "¿Cómo debo comportarme con el personal administrativo?", "clave": ["respet"]},
    {"p": "¿De qué color es el uniforme de gala?",               "clave": ["no encontré"]},
    {"p": "¿Cuánto cuesta el estacionamiento?",                  "clave": ["no encontré"]},
    {"p": "¿Hay clases de natación?",                            "clave": ["no encontré"]},
]

def evaluar(funcion_respuesta, mostrar=False):
    aciertos = 0
    for item in preguntas_eval:
        respuesta = funcion_respuesta(item["p"])
        acierto = any(c in respuesta.lower() for c in item["clave"])
        aciertos += acierto
        if mostrar:
            print(("CORRECTO = " if acierto else "INCORRECTO = "), item["p"], "->", respuesta)
    return aciertos / len(preguntas_eval)


### **LLAMA 8B SIN RAG Y CON RAG**

In [54]:
# Evaluar Llama 8B sin RAG y con RAG
def responder_sin_rag(pregunta):
    r = client.chat(
        model=MODELO,
        messages=[{"role": "user", "content": f"Responde sobre el reglamento escolar: {pregunta} Respuesta breve."}]
    )
    return r.message.content.strip()

precision_sin_rag = evaluar(responder_sin_rag)
precision_con_rag = evaluar(responder_rag, mostrar=True)

print(f"\nLlama 8B sin RAG: {precision_sin_rag:.0%}")
print(f"Llama 8B + RAG:   {precision_con_rag:.0%}")

CORRECTO =  ¿Qué pasa si llego tarde a la escuela? -> Los estudiantes deben respetar los horarios de entrada y salida establecidos por la institución. Los retardos deberán justificarse ante las autoridades escolares.
CORRECTO =  ¿Puedo meter sustancias ilegales al plantel? -> No, está prohibido introducir sustancias ilegales a la institución. (Revisa el punto 1 del reglamento)
CORRECTO =  ¿Es obligatorio llevar la credencial? -> Sí, es obligatorio llevar la credencial escolar dentro de las instalaciones. Los estudiantes deben presentarla cuando sea solicitada por el personal autorizado.
CORRECTO =  ¿Qué porcentaje de asistencia me piden? -> Un mínimo de 80%.
CORRECTO =  ¿Cuál es la calificación mínima para aprobar? -> La calificación mínima para aprobar es 6. Se requiere una calificación de 6 en adelante para aprobar.
INCORRECTO =  ¿Puedo contestar mensajes en clase con mi celular? -> No encontré esa información en el reglamento.
CORRECTO =  Si dañé una butaca, ¿quién la paga? -> Los d

## **FINE-TUNING CON LORA**
Se ajusta Llama 3.2 de 1B con LoRA para que aprenda a responder de forma breve usando el contexto que recibe y rechazar cuando el contexto no contiene la respuesta.

### **CARGAR EL MODELO BASE**

In [55]:
 #Cargar el modelo base de Llama 1B y su tokenizer
import logging
logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.utils import logging
logging.set_verbosity_error()

modelo_base = "unsloth/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")
print("Modelo base cargado:", modelo_base)

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

Modelo base cargado: unsloth/Llama-3.2-1B-Instruct


### **FUNCIÓN DE GENERACIÓN**

In [57]:
# Definir la función de generación de texto
def generar_respuesta(modelo_a_usar, prompt, max_new_tokens=80):
    entrada = tokenizer(prompt, return_tensors="pt").to(modelo_a_usar.device)
    salida = modelo_a_usar.generate(
        **entrada,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
        no_repeat_ngram_size=3,
    )
    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]
    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)
    return texto_generado.split("\n")[0].strip()

### **PREPARAR EL DATASET DE ENTRENAMIENTO**


In [58]:
# Construir el dataset con el formato Contexto / Pregunta / Respuesta
from datasets import Dataset

def a_texto(indices, pregunta, respuesta):
    contexto = " ".join(documentos[i] for i in indices)
    return {"texto": f"Contexto: {contexto}\nPregunta: {pregunta}\nRespuesta: {respuesta}"}

ejemplos = [
    a_texto([2, 5], "¿Qué debo hacer si llego tarde?", "Debes justificar el retardo ante las autoridades escolares."),
    a_texto([0, 4], "¿Debo traer mi credencial?", "Sí, debes portarla dentro de la institución y presentarla cuando el personal la solicite."),
    a_texto([1, 6], "¿Se permite traer alcohol a la escuela?", "No, está prohibido introducir bebidas alcohólicas, sustancias ilegales u objetos de riesgo."),
    a_texto([5, 3], "¿Cuánta asistencia necesito para presentar examen?", "Necesitas al menos 80% de asistencia para tener derecho a examen ordinario."),
    a_texto([6, 2], "¿Con cuánto apruebo una materia?", "Con calificación de 6 o más en la escala de 0 a 10."),
    a_texto([4, 1], "¿Puedo usar el celular en clase?", "Solo si el docente te lo autoriza."),
    a_texto([7, 0], "¿Qué pasa si rompo una silla del salón?", "Debes repararla o pagarla."),
    a_texto([8, 3], "¿Cuánto tiempo tengo para justificar una falta?", "Tienes 3 días hábiles para presentar el comprobante."),
    a_texto([9, 5], "¿Qué pasa si me sorprenden copiando?", "Se anula tu examen desde la primera falta."),
    a_texto([3, 7], "¿Cómo debo tratar a mis maestros?", "Con respeto; las faltas de respeto serán sancionadas."),
    a_texto([0, 2], "¿Quién puede pedirme la credencial?", "El personal autorizado de la institución."),
    a_texto([2, 8], "¿Hay horarios de entrada y salida?", "Sí, debes respetar los horarios de entrada y salida establecidos por la institución."),
    a_texto([0, 1], "¿De qué color es el uniforme?", MENSAJE_NO_SABE),
    a_texto([4, 6], "¿Cuánto cuesta la inscripción?", MENSAJE_NO_SABE),
    a_texto([3, 5], "¿A qué hora abre la cafetería?", MENSAJE_NO_SABE),
    a_texto([7, 9], "¿Hay becas disponibles?", MENSAJE_NO_SABE),
]

dataset = Dataset.from_list(ejemplos)
split = dataset.train_test_split(test_size=0.25, seed=42)
print("Ejemplos de entrenamiento:", len(split["train"]), "| validación:", len(split["test"]))
print("\nEjemplo:\n", ejemplos[0]["texto"])

Ejemplos de entrenamiento: 12 | validación: 4

Ejemplo:
 Contexto: Los estudiantes deben respetar los horarios de entrada y salida establecidos por la institución. Los retardos deberán justificarse ante las autoridades escolares. Se requiere un mínimo de 80% de asistencia para tener derecho a examen ordinario.
Pregunta: ¿Qué debo hacer si llego tarde?
Respuesta: Debes justificar el retardo ante las autoridades escolares.


### **CONFIGURAR Y APLICAR LORA**


In [59]:
# Configurar LoRA y aplicarlo al modelo base
!pip uninstall -y torchao > /dev/null 2>&1  # evita un error de peft si Colab trae un torchao viejo

from peft import LoraConfig, get_peft_model
from transformers import set_seed
set_seed(42)

config_lora = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.0,
    task_type="CAUSAL_LM"
)

modelo_lora = get_peft_model(modelo, config_lora)
modelo_lora.print_trainable_parameters()


trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


### **ENTRENAR CON LORA**

Medimos la **pérdida de validación antes y después** del entrenamiento sobre ejemplos que el modelo no usa para entrenar. Es una métrica más confiable que la pérdida de entrenamiento, que con pocos ejemplos solo muestra memorización.

In [60]:
# Configurar el entrenador, medir la pérdida antes y ejecutar el fine-tuning
from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(
    output_dir="/content/resultados",
    num_train_epochs=8,
    per_device_train_batch_size=4,
    learning_rate=2e-4,
    logging_steps=1,
    dataset_text_field="texto",
    max_length=256,
    eval_strategy="epoch",
    report_to="none",
    fp16=True,   # la T4 no soporta bf16
    bf16=False,
)

trainer = SFTTrainer(
    model=modelo_lora,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    args=config_entrenamiento,
)

perdida_antes = trainer.evaluate()["eval_loss"]
trainer.train()
perdida_despues = trainer.evaluate()["eval_loss"]

# Dejar el modelo listo para generar texto
modelo_lora.eval()
modelo_lora.config.use_cache = True
if hasattr(modelo_lora, "gradient_checkpointing_disable"):
    modelo_lora.gradient_checkpointing_disable()

print(f"Pérdida de validación antes:   {perdida_antes:.3f}")
print(f"Pérdida de validación después: {perdida_despues:.3f}")
print(f"Reducción: {(1 - perdida_despues/perdida_antes) * 100:.0f}%")

Adding EOS to train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/12 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/4 [00:00<?, ? examples/s]

{'eval_loss': '2.438', 'eval_model_preparation_time': '0.0155', 'eval_runtime': '0.5024', 'eval_samples_per_second': '7.962', 'eval_steps_per_second': '1.991', 'eval_entropy': '2.328', 'eval_mean_token_accuracy': '0.4847', 'eval_num_tokens': '0', 'epoch': 0}
{'loss': '2.148', 'grad_norm': '0.8075', 'learning_rate': '0.0002', 'entropy': '2.132', 'mean_token_accuracy': '0.5416', 'num_tokens': '401', 'epoch': '0.3333'}
{'loss': '2.309', 'grad_norm': '0.9154', 'learning_rate': '0.0001917', 'entropy': '2.317', 'mean_token_accuracy': '0.4986', 'num_tokens': '754', 'epoch': '0.6667'}
{'loss': '2.285', 'grad_norm': '0.9888', 'learning_rate': '0.0001833', 'entropy': '2.109', 'mean_token_accuracy': '0.5067', 'num_tokens': '1129', 'epoch': '1'}
{'eval_loss': '2.368', 'eval_model_preparation_time': '0.0155', 'eval_runtime': '0.0766', 'eval_samples_per_second': '52.24', 'eval_steps_per_second': '13.06', 'eval_entropy': '2.326', 'eval_mean_token_accuracy': '0.493', 'eval_num_tokens': '1129', 'epoch'

## **COMPARACIÓN DE LAS TRES VERSIONES**

Ahora usamos el modelo ajustado (1B + LoRA) con los mismos fragmentos recuperados por RAG y lo comparamos contra las dos líneas base.

In [61]:
# Definir el asistente con el modelo ajustado y evaluar
def responder_lora(pregunta, k=K):
    contexto = " ".join(buscar_fragmentos(pregunta, k))
    prompt = f"Contexto: {contexto}\nPregunta: {pregunta}\nRespuesta:"
    return generar_respuesta(modelo_lora, prompt)

precision_lora = evaluar(responder_lora, mostrar=True)

print("\n--- RESULTADOS ---")
print(f"Llama 8B sin RAG:        {precision_sin_rag:.0%}")
print(f"Llama 8B + RAG:          {precision_con_rag:.0%}")
print(f"Llama 1B + LoRA + RAG:   {precision_lora:.0%}")
print(f"Pérdida de validación:   {perdida_antes:.3f} -> {perdida_despues:.3f}")

CORRECTO =  ¿Qué pasa si llego tarde a la escuela? -> Debe justificar su retardo ante las instituciones escolares y presentar el comprobante dentro de los 3 dias.
CORRECTO =  ¿Puedo meter sustancias ilegales al plantel? -> No, no se permite introducir sustancias prohibidas en el campus.
INCORRECTO =  ¿Es obligatorio llevar la credencial? -> Sí, es obligatorio.
CORRECTO =  ¿Qué porcentaje de asistencia me piden? -> Se pide un mínimo del 80%.
INCORRECTO =  ¿Cuál es la calificación mínima para aprobar? -> 80%.
INCORRECTO =  ¿Puedo contestar mensajes en clase con mi celular? -> No, no se permite usar teléfones celulares en clase.
CORRECTO =  Si dañé una butaca, ¿quién la paga? -> El responsable de la butaca deberá pagar por el daño.
CORRECTO =  ¿En cuántos días entrego un justificante médico? -> 3.
CORRECTO =  ¿Qué sanción hay por copiar en un examen? -> La primera falta es anular el examén.
CORRECTO =  ¿Cómo debo comportarme con el personal administrativo? -> Debo mantener un comportamien

In [62]:
# Elegir la mejor versión y publicarla con Gradio
import gradio as gr

funcion_final = responder_rag

demo = gr.Interface(
    fn=funcion_final,
    inputs=gr.Textbox(label="Pregunta sobre el reglamento"),
    outputs=gr.Textbox(label="Respuesta"),
    title="Asistente del reglamento escolar",
)
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f2ef930e1369bbc38a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
